### RAG Pipeline : Data Ingestion to Vector DB

In [1]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Saif\AppData\Local\Temp\ipykernel_14912\2096081597.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
#Read all the pdfs inside the directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} files to process")

    for pdf_file in pdf_files:
        print(f"\nprocessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
        #add source info to metadata
            for doc in documents:
                doc.metadata['source'] = pdf_file.name
                doc.metadata['type']= ".pdf"

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents




all_pdf_docs = process_all_pdfs("../data")

found 4 files to process

processing attention.pdf
  ✓ Loaded 3 pages

processing embeddings.pdf
  ✓ Loaded 3 pages

processing objectdetection.pdf
  ✓ Loaded 3 pages

processing proposal.pdf
  ✓ Loaded 3 pages

Total documents loaded: 12


In [3]:
all_pdf_docs

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-09-21T12:38:07-06:00', 'author': 'Saif mukadam', 'moddate': '2026-09-21T12:38:07-06:00', 'source': 'attention.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'type': '.pdf'}, page_content='Attention Mechanism in Deep Learning \nIntroduction \nThe Attention Mechanism is one of the most important innovations in deep learning. Instead \nof treating every input equally, attention allows a model to focus on the most relevant parts \nof the input while making predictions. \nOriginally introduced for Neural Machine Translation, attention is now the foundation of \nmodern Large Language Models (LLMs) such as GPT, Gemini, Claude, and Llama. \n \nWhy Attention? \nTraditional Recurrent Neural Networks (RNNs) struggled with long sequences because \ninformation from earlier words was gradually forgotten. \nExample: \n"The cat that was sitting on the roof for several hours finally j

In [4]:
def text_splitting(documents, chunk_size = 100, chunk_overlap = 20):
    """Creating Chunks"""
    document_splitter = RecursiveCharacterTextSplitter( chunk_size = chunk_size,
    chunk_overlap = chunk_overlap, length_function = len, separators = ["\n\n", "\n", " ",","])

    split_docs = document_splitter.split_documents(all_pdf_docs)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Metadata: {split_docs[0].metadata}")
        print(f"Content: {split_docs[0].page_content[:200]}...")

    return split_docs
chunks = text_splitting(all_pdf_docs)


Split 12 documents into 97 chunks

Example chunk:
Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-09-21T12:38:07-06:00', 'author': 'Saif mukadam', 'moddate': '2026-09-21T12:38:07-06:00', 'source': 'attention.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'type': '.pdf'}
Content: Attention Mechanism in Deep Learning 
Introduction...


### Embedding and VectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    def __init__(self , model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts : List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("ModelNotLoaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embeddings = EmbeddingManager()
embeddings

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


### VectorStore 

In [7]:
class VectorStore():
    def __init__(self, collection_name: str = "pdf_documents", persistent_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.collection = None
        self.client = None
        self._initialize_vectorstore()

    def _initialize_vectorstore(self):
        """Initialize Chromadb client and collection"""
        try:
            os.makedirs(self.persistent_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Prepare data for chromadb"""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        ids = []
        documents_text = []
        metadatas = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")

            metadata = dict(doc.metadata)
            metadata["id"] = i
            metadata["content"] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        if not documents:
            return

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 194


In [8]:
chunks


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-09-21T12:38:07-06:00', 'author': 'Saif mukadam', 'moddate': '2026-09-21T12:38:07-06:00', 'source': 'attention.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'type': '.pdf'}, page_content='Attention Mechanism in Deep Learning \nIntroduction'),
 Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-09-21T12:38:07-06:00', 'author': 'Saif mukadam', 'moddate': '2026-09-21T12:38:07-06:00', 'source': 'attention.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'type': '.pdf'}, page_content='The Attention Mechanism is one of the most important innovations in deep learning. Instead'),
 Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-09-21T12:38:07-06:00', 'author': 'Saif mukadam', 'moddate': '2026-09-21T12:38:07-06:00', 'source': 'attention.pdf', 'total

In [9]:
#Convert text to embeddings
texts = [doc.page_content for doc in chunks]

#generate embeddings for the chunks
embeddings_array = embeddings.generate_embeddings(texts)


#Add documents to vector store
vectorstore.add_documents(chunks, embeddings_array)
vectorstore

Generating embeddings for 97 texts...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Generated embeddings with shape: (97, 384)
Successfully added 97 documents to vector store
Total documents in collection: 291


#### Retriever Pipeline from VectorStore

In [10]:
class Retriever:
    def __init__(self, vectorstore: VectorStore, embeddings: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embeddings = embeddings

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float= 0.0) -> List[dict[str, Any]]:
        """Retrieve relevant documents based on query"""
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #Generate Query Embedding
        query_embedding = self.embeddings.generate_embeddings([query])[0]

        #Search in VectorStore
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            #process reults
            retrieved_docs =[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                #Convert distance to cosine similarity score
                for i, (doc_id, metadata, distance, document) in enumerate(zip(ids, metadatas, distances, documents)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                        print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                    else:
                        print("No documents found")
            
            return retrieved_docs


                

        
        except Exception as e:
            print(f"Error querying vector store: {e}")
            raise

retriever = Retriever(vectorstore, embeddings)
retriever


In [11]:
retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)


[{'id': 'doc_0027f694_2',
  'content': 'of treating every input equally, attention allows a model to focus on the most relevant parts',
  'metadata': {'page': 0,
   'content': 93,
   'producer': 'Microsoft® Word 2021',
   'total_pages': 3,
   'author': 'Saif mukadam',
   'source': 'attention.pdf',
   'page_label': '1',
   'type': '.pdf',
   'creationdate': '2026-09-21T12:38:07-06:00',
   'id': 2,
   'creator': 'Microsoft® Word 2021',
   'moddate': '2026-09-21T12:38:07-06:00'},
  'similarity_score': 0.2327185869216919,
  'distance': 0.7672814130783081,
  'rank': 1},
 {'id': 'doc_d4e90ad5_2',
  'content': 'of treating every input equally, attention allows a model to focus on the most relevant parts',
  'metadata': {'content': 93,
   'producer': 'Microsoft® Word 2021',
   'type': '.pdf',
   'page': 0,
   'page_label': '1',
   'author': 'Saif mukadam',
   'total_pages': 3,
   'creationdate': '2026-09-21T12:38:07-06:00',
   'creator': 'Microsoft® Word 2021',
   'moddate': '2026-09-21T12:38:

### Simple RAG Pipeline

In [12]:
import os 
from dotenv import load_dotenv
load_dotenv()
gorq_api_key = os.getenv("GROQ_API_KEY")

In [13]:
from langchain_groq import ChatGroq

In [14]:
llm = ChatGroq(
    groq_api_key=gorq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.0,
    max_tokens=1024,
)

#2. Simple RAG FUnction

def rag_simple(query, retriever,llm, top_k=3):
    """retrieve the context"""
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n". join([doc['content'] for doc in results]) if results else ""
    

    #Generate answer using groq  LLM
    prompt = f"Answer the following question based on the context below:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"

    response = llm.invoke([prompt.format(query = query, context = context)])
    return response.content
    

In [15]:
answer = rag_simple("What is attention is all you need", retriever, llm)

Retrieving documents for query: 'What is attention is all you need'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)


### Enhanced RAG pipeline



In [16]:
def enhanced_rag(query, retriever, llm, top_k= 5, min_score = 0.5, return_context = False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k= top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    context ="\n\n".join([ doc['content'] for doc in results])
    sources = [{
        'source': (doc['metadata'] or {}).get('source_file', (doc['metadata'] or {}).get('source', 'unknown')),
        'page': (doc['metadata'] or {}).get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)

    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence,
    }

    if return_context:
        output['context'] = context


    return output

result = enhanced_rag("Object Detection", retriever, llm, top_k=5, min_score=0.5, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'Object Detection'
Top K: 5, Score threshold: 0.5
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)
Answer: Object Detection is a computer‑vision task that identifies and localizes objects within an image.
Sources: [{'source': 'objectdetection.pdf', 'page': 0, 'score': 0.6681578755378723, 'preview': 'Object Detection \nIntroduction...'}, {'source': 'objectdetection.pdf', 'page': 0, 'score': 0.6681578755378723, 'preview': 'Object Detection \nIntroduction...'}, {'source': 'objectdetection.pdf', 'page': 0, 'score': 0.6681578755378723, 'preview': 'Object Detection \nIntroduction...'}, {'source': 'objectdetection.pdf', 'page': 0, 'score': 0.5311872959136963, 'preview': 'Object Detection is a computer vision task that identifies objects within an image and...'}, {'source': 'objectdetection.pdf', 'page': 0, 'score': 0.5311872959136963, 'preview': 'Objec

In [20]:
# Advance RAG Pipeline with streaming, citations, history, summarization.
import time

class AdvanceRagPipeline():
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""

            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

             # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvanceRagPipeline(retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
of treating every input equally, attention allows a model to focus on the most relevant parts

of treating every input equally, attention allows a model to focus on the most relevant parts

of treating every input equally, attention allows a model to focus on the most relevant parts

Question: what is attention is all you need

Answer:

Final Answer: “Attention Is All You Need” is the title of the 2017 paper that introduced the Transformer architecture. It shows that a model can learn sequence relationships using only self‑attention mechanisms—without recurrence or convolution—achieving state‑of‑the‑art performance on tasks like machine translation.

Citations:
[1] attention.pdf (page 0)
[2] attention.pdf (page 0)
[3] attention.pdf (page 0)
